# Notebook 3 — Resume Section Classifier with a BiLSTM

**Project:** GenAI Resume Analyser
**Input:** one line/sentence from a resume → **Output:** which section it belongs to (experience, education, project, skills, other)

## Syllabus topics this notebook covers
1. **RNN / LSTM** — a bidirectional LSTM reads each sentence token-by-token, forward and backward
2. **Embeddings** — a trainable word-embedding matrix (different from the sentence-transformer embeddings used for the match score)
3. **NLP preprocessing** — tokenization, vocabulary building, padding
4. **Evaluation + class imbalance** — macro-F1 and a class-weighted loss

## Dataset: chingkuangkam "Resume Text Classification Dataset" (format verified against the real files)
500 resumes as txt files; **every line is `position \t section \t text`** — 40,004 labelled lines total.
Real section labels: **experience 46%, others 26%, education 15%, knowledge 10%, project 2%**.
`knowledge` is the TECHNICAL SKILLS block, so it maps to our `skills` section. Bonus for the viva: the
`position` column marks contact-info lines as `meta`, which supports the app's "names are stripped before scoring" story.

## How to run
1. Colab: `Runtime` → `Change runtime type` → **T4 GPU** → Save (CPU works too, just slower)
2. `Runtime` → `Run all` — the dataset downloads automatically; **no manual upload needed** if your Kaggle secrets are set. Without secrets, the notebook asks you to upload the ZIP from the Kaggle page.
3. Training: ~3–5 min on T4 (~15 min CPU). The last cell downloads `section_classifier_artifacts.zip` → move into `notebooks/artifacts/`

## What good results look like
- Accuracy ≈ 0.80–0.92, macro-F1 ≈ 0.70–0.85
- `experience` and `education` score high; `project` has only ~700 of 40,000 lines, so its recall is naturally the weakest — the class-weighted loss softens (not cures) that
- If macro-F1 < 0.55 → check the "Label mapping used:" printout below matches the five canonical sections

In [ ]:
# Nothing heavy to install — torch ships with Colab; we just need the Kaggle downloader and plotting libs
!pip -q install "kagglehub" "scikit-learn>=1.4" matplotlib seaborn

In [ ]:
# Imports + global seed so weight init, shuffling and the split are all reproducible
import os, re, json, glob, shutil, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 1. Get the dataset
**Kaggle way (automatic):** Colab secrets `KAGGLE_USERNAME` + `KAGGLE_KEY` (🔑 sidebar) — the notebook downloads and finds the files itself.
**Manual way (no secrets):** on Kaggle open [resume-text-classification-dataset](https://www.kaggle.com/datasets/chingkuangkam/resume-text-classification-dataset) → Download → you get a ZIP; run this cell and pick that ZIP in the upload dialog.

In [ ]:
# Download the dataset via kagglehub (a folder of 500 txt files); fall back to uploading the Kaggle ZIP manually
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Kaggle credentials loaded from Colab secrets")
    import kagglehub
    base = kagglehub.dataset_download("chingkuangkam/resume-text-classification-dataset")
    dirs = sorted(d for d in glob.glob(os.path.join(base, "**", "resume_500"), recursive=True) if os.path.isdir(d))
    assert dirs, "resume_500 folder not found in the downloaded dataset"
    DATA_PATH = dirs[0]
except Exception as e:
    print(f"Kaggle download unavailable ({e}) -> manual upload mode: pick the ZIP you downloaded from Kaggle")
    from google.colab import files
    import zipfile
    up = files.upload()
    with zipfile.ZipFile(list(up.keys())[0]) as z:
        z.extractall("/content/manual_nb3")
    dirs = sorted(d for d in glob.glob("/content/manual_nb3/**/resume_500", recursive=True) if os.path.isdir(d))
    assert dirs, "Uploaded ZIP did not contain a resume_500 folder"
    DATA_PATH = dirs[0]
print("Using dataset folder:", DATA_PATH)

In [ ]:
# Read all 500 resume files: every line is position \t section \t text (verified real format)
rows = []
for fp in sorted(glob.glob(os.path.join(DATA_PATH, "*.txt"))):
    for ln in open(fp, encoding="utf-8-sig", errors="replace").read().splitlines():
        parts = ln.split("\t", 2)                  # at most 2 splits: the text itself may contain tabs
        if len(parts) == 3 and parts[2].strip():
            rows.append({"position": parts[0].strip().lower(),
                         "label": parts[1].strip().lower(),
                         "text": parts[2].strip()})
df = pd.DataFrame(rows)
df["label"] = df["label"].replace({"otherse": "others"})   # one typo row in the real data
print("Rows:", len(df), "| resume files:", len(glob.glob(os.path.join(DATA_PATH, "*.txt"))))
print("\nRaw label distribution:\n", df["label"].value_counts())
print("\nSample rows:\n", df[["position", "label", "text"]].head(4).to_string(index=False))

## 2. Label mapping (fixed — verified against all 500 real files)
The dataset's real labels map onto our five canonical dashboard sections as:
`experience→experience`, `education→education`, `knowledge→skills` (it IS the technical-skills block),
`project→project`, `others→other`. The printout below is checked against these expectations.

In [ ]:
# Fixed mapping for the dataset's verified labels; a keyword fallback guards against any surprise label
LABEL_MAP = {"experience": "experience", "education": "education", "knowledge": "skills",
             "project": "project", "others": "other"}
def auto_map(lbl):
    s = str(lbl).lower()
    for key, canon in LABEL_MAP.items():
        if key in s: return canon
    if "edu" in s: return "education"
    if "exp" in s or "work" in s or "employ" in s: return "experience"
    if "skill" in s or "know" in s: return "skills"
    if "proj" in s: return "project"
    return "other"
raw_labels = sorted(df["label"].astype(str).unique())
print("Label mapping used:")
for l in raw_labels: print(f"  {l!r:20s} -> {auto_map(l)}")
df["section"] = df["label"].map(auto_map)
sections = ["experience", "education", "project", "skills", "other"]
present = set(df["section"].unique())
assert present <= set(sections), f"Unmapped sections appeared: {present - set(sections)} - extend LABEL_MAP above"
df = df[["text", "section", "position"]].dropna().reset_index(drop=True)
print("\nFinal distribution:\n", df["section"].value_counts())

In [ ]:
# Clean + tokenize: simple regex word tokenizer keeps this notebook self-contained (no external tokenizer dependency)
def tokenize(text):
    return re.findall(r"[a-z]+", str(text).lower())
df["tokens"] = df["text"].apply(tokenize)
df = df[df["tokens"].str.len() > 0].reset_index(drop=True)
print("Example:", df["text"].iloc[0][:80], "->", df["tokens"].iloc[0][:8])

In [ ]:
# Stratified split, then build the vocabulary from TRAIN ONLY (vocabulary leakage would inflate test scores)
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["section"], random_state=SEED)
vocab = {"<pad>": 0, "<unk>": 1}
for toks in train_df["tokens"]:
    for t in toks:
        if t not in vocab:
            vocab[t] = len(vocab)
print(f"Train: {len(train_df)}  Test: {len(test_df)}  Vocab size: {len(vocab)}")

## 3. The BiLSTM model
An LSTM processes the sentence token-by-token, keeping a hidden "memory" updated at each step. **Bidirectional** means two LSTMs read the sentence forward and backward and their outputs are concatenated, so "Bachelor of" and "of Science" both see their right-hand context. The embedding layer maps each word id to a dense 100-d vector that is *learned during training* — that is the word-embeddings syllabus topic.

In [ ]:
# Encode sentences as padded id sequences + integer labels, wrapped in TensorDatasets and DataLoaders
MAX_LEN, BATCH = 50, 64
def encode(toks):
    ids = [vocab.get(t, 1) for t in toks[:MAX_LEN]]
    return ids + [0] * (MAX_LEN - len(ids))
sec2id = {s: i for i, s in enumerate(sections)}
def make_loader(frame, shuffle):
    X = torch.tensor([encode(t) for t in frame["tokens"]], dtype=torch.long)
    y = torch.tensor([sec2id[s] for s in frame["section"]], dtype=torch.long)
    return DataLoader(TensorDataset(X, y), batch_size=BATCH, shuffle=shuffle)
train_loader, test_loader = make_loader(train_df, True), make_loader(test_df, False)

In [ ]:
# Define the BiLSTM: Embedding -> BiLSTM(128 hidden, 2 layers) -> dropout -> Linear over the 5 sections
class SectionBiLSTM(nn.Module):
    def __init__(self, vocab_size, n_classes, emb_dim=100, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden, num_layers=2, bidirectional=True, batch_first=True, dropout=0.3)
        self.drop = nn.Dropout(0.3)
        self.out = nn.Linear(hidden * 2, n_classes)
    def forward(self, x):
        emb = self.emb(x)                                   # (batch, time, emb_dim) - the learned word embeddings
        _, (h, _) = self.lstm(emb)                          # h: (layers*2, batch, hidden)
        feat = torch.cat([h[-2], h[-1]], dim=1)             # concat last forward + backward hidden states
        return self.out(self.drop(feat))

model = SectionBiLSTM(len(vocab), len(sections)).to(DEVICE)
print("Parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
# Training loop: Adam + cross-entropy, evaluate every epoch, keep the best checkpoint by validation macro-F1
os.makedirs("/content/checkpoints/section_bilstm", exist_ok=True)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
counts = np.bincount([sec2id[s] for s in train_df["section"]], minlength=len(sections))  # project is only ~2% of lines
class_w = torch.tensor(len(train_df) / (len(sections) * counts), dtype=torch.float32).to(DEVICE)
print("Class counts:", dict(zip(sections, counts.tolist())), "| weights:", [round(w, 2) for w in class_w.tolist()])
loss_fn = nn.CrossEntropyLoss(weight=class_w)  # inverse-frequency weights compensate the class imbalance
EPOCHS = 10
history = []
best_f1 = 0.0
for ep in range(1, EPOCHS + 1):
    model.train(); total = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward(); opt.step()
        total += loss.item() * len(xb)
    model.eval(); yt, yp = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            pr = model(xb.to(DEVICE)).argmax(dim=1).cpu()
            yt += yb.tolist(); yp += pr.tolist()
    acc = accuracy_score(yt, yp); f1 = f1_score(yt, yp, average="macro")
    history.append((ep, total / len(train_df), acc, f1))
    print(f"Epoch {ep:2d}  loss={total/len(train_df):.4f}  acc={acc:.4f}  macro-F1={f1:.4f}")
    if f1 > best_f1:                                    # checkpoint: keep only the best epoch's weights
        best_f1 = f1
        torch.save(model.state_dict(), "/content/checkpoints/section_bilstm/best.pt")

In [ ]:
# Plot the training curves (loss and macro-F1 per epoch) and save them for the report
h = pd.DataFrame(history, columns=["epoch", "loss", "acc", "f1"])
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h.epoch, h.loss, marker="o"); ax[0].set_title("Training loss"); ax[0].set_xlabel("Epoch")
ax[1].plot(h.epoch, h.acc, marker="o", label="accuracy"); ax[1].plot(h.epoch, h.f1, marker="o", label="macro-F1")
ax[1].set_title("Validation metrics"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout(); os.makedirs("/content/artifacts/section_classifier", exist_ok=True)
plt.savefig("/content/artifacts/section_classifier/training_curves.png", dpi=150); plt.show()

In [ ]:
# Load the best checkpoint, print the per-class report and draw the confusion matrix
model.load_state_dict(torch.load("/content/checkpoints/section_bilstm/best.pt"))
model.eval()
yt, yp = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        yp += model(xb.to(DEVICE)).argmax(dim=1).cpu().tolist(); yt += yb.tolist()
final_acc, final_f1 = accuracy_score(yt, yp), f1_score(yt, yp, average="macro")
print(classification_report(yt, yp, labels=list(range(len(sections))), target_names=sections, zero_division=0))
cm = confusion_matrix(yt, yp)
plt.figure(figsize=(6.5, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", xticklabels=sections, yticklabels=sections)
plt.title("Confusion matrix - BiLSTM section classifier")
plt.ylabel("True section"); plt.xlabel("Predicted section"); plt.tight_layout()
plt.savefig("/content/artifacts/section_classifier/cm_bilstm.png", dpi=150); plt.show()

In [ ]:
# Save artifacts: model weights, vocabulary, label map, metrics JSON - then zip for download
out = "/content/artifacts/section_classifier"
shutil.copy("/content/checkpoints/section_bilstm/best.pt", out + "/bilstm_best.pt")
with open(out + "/vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)
with open(out + "/labels.json", "w") as f:
    json.dump({"sections": sections, "section2id": sec2id, "raw_label_map": LABEL_MAP,
               "max_len": MAX_LEN}, f, indent=2)
with open(out + "/metrics.json", "w") as f:
    json.dump({"task": "section_classification", "model": "bilstm-2layer-bidirectional",
               "n_train": len(train_df), "n_test": len(test_df), "vocab_size": len(vocab),
               "accuracy": round(final_acc, 4), "macro_f1": round(final_f1, 4)}, f, indent=2)
shutil.make_archive("/content/section_classifier_artifacts", "zip", out)
print("Zipped:", os.path.getsize("/content/section_classifier_artifacts.zip") // 1024, "KB")

In [ ]:
# Trigger the browser download - move the zip into D:\resume-analyser\notebooks\artifacts\
from google.colab import files
files.download("/content/section_classifier_artifacts.zip")

## Explain this to my examiner

**What the model is.** A 2-layer **bidirectional LSTM** over trainable word embeddings. Each of the ~40,000 labelled resume lines is tokenized, converted to word-id sequences padded to 50 tokens, embedded into 100-d vectors, then read by two stacked LSTMs in both directions; the final forward and backward hidden states are concatenated and classified by a linear layer into 5 sections. Trained for 10 epochs with Adam (lr 1e-3) and a **class-weighted cross-entropy loss**.

**Why bidirectional?** Section membership depends on both left and right context: "Bachelor of Technology" is identified partly by what *follows* the institution name. A unidirectional LSTM only sees the past; the backward pass supplies the future context.

**What the metrics mean.** Accuracy is the share of lines labeled correctly; **macro-F1** averages F1 per section unweighted. That matters here because `experience` alone is 46% of lines — a lazy majority-class model would already score 46% accuracy while being useless; macro-F1 exposes that. The confusion matrix shows which sections get mixed up.

**Class imbalance handling.** `project` has only ~713 of 40,004 lines (~2%). I compensated with inverse-frequency class weights in the loss, n/(classes×count) — a standard technique — and report macro-F1 so `project` counts equally with `experience`.

**Privacy angle.** The dataset's `position` column marks contact lines (name, phone, email) as `meta`, and those lines carry section label `others`. In the final app the NER model's `Name` entities are likewise stripped before any scoring — no name or gender ever influences the match score.

**Syllabus topics covered.** RNN/LSTM architecture (gates, hidden state, bidirectionality), word embeddings (a trained embedding matrix, distinct from the pretrained sentence-transformer embeddings used for the match score), NLP preprocessing (tokenization, vocabulary built from train data only, padding), class imbalance handling, and multi-class evaluation (macro-F1, confusion matrix).

**Why not a transformer here?** A transformer would likely score higher, but this component exists to cover the RNN/LSTM syllabus topic and runs in milliseconds per line in the web dashboard — a deliberate engineering trade-off worth stating openly.

## Troubleshooting
- **`resume_500 folder not found`** → the Kaggle layout changed; run `!find /root/.cache/kagglehub -name "*.txt" | head -3` and adjust the folder name in the download cell.
- **Project recall very low (or 0)** → partly expected (2% of data); confirm the class-weight lines ran — the training cell prints the weight vector (project's weight should be the largest).
- **Curves diverge or NaN** → rerun all cells from the top in order; the seed makes a clean run deterministic.
- **CPU runtime slow** → ~15 min instead of ~4; T4 recommended but not required.